# Silver layer

## Customer additional information (location)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, trim, upper, now, isnull, current_date, max, min, isnotnull, length, lag, date_add, lead, isnull, ifnull
from pyspark.sql.types import *

In [0]:
df = spark.read.table("db_project.bronze.erp_loc_a101")
df.display()

## Modify CID

Customer id provided with "AW-00011049" expression is probably corelated to cst_key (customer key) in crm_cust_info with "AW00011000" expression \
* Remove "-"

In [0]:
df = df.withColumn("CID", trim(F.regexp_replace(df. CID, "-", "")))
df.display()

In [0]:
test_id = df.select(
    min(length(df.CID)).alias("min CID length"),
    max(length(df.CID)).alias("max CID length"),
    F.avg(length(df.CID)).alias("avg CID length")
)
test_id.display()

## Distinct countries

In [0]:
df.select("CNTRY").distinct().display()

Countries provided have duplicates (USA/US/United States) and lacking data. \
To continue, following logic will be applied:
* Only countries full names
* Unknown will be relaced with "n/a" 

In [0]:
df = df.withColumn("CNTRY", 
              F.when(col("CNTRY").isin("USA", "US"), "United States")
              .when(col("CNTRY") == "DE", "Germany")
              .when((trim(col("CNTRY")) == "") | (col("CNTRY").isNull()), "n/a")
              .otherwise(col("CNTRY"))
)
df.display()


# Write table silver.erp_loc_a101

Everything looks ok, so save DataFrame into Delta Table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", True).format("delta").saveAsTable("db_project.silver.erp_loc_a101")